# Analyzing Wikipedia Pages

In this project, we will analyze Wikipedia pages to gain insights into the content and structure of the articles. We will use Python and various libraries to scrape and process the data, and then visualize the results to better understand the Wikipedia ecosystem. We will implement a quick script command to collect 500 MB worth of data for our processing. For reference, the below script was generated by Claude AI as the course hasn't reached webscraping as of yet. Using the script, we generated 1359 files, which are otherwise found in the `wiki` folder in this directory.

In [2]:
import os, time, requests
from urllib.parse import quote

# --- settings ---
OUT_DIR   = 'wiki'
TARGET_MB = 200      # stop once the folder reaches this size
DELAY     = 0.2      # seconds between requests
# ----------------

HEADERS = {'User-Agent': 'grep-practice/1.0 (learning project)'}
API = 'https://en.wikipedia.org/w/api.php'
os.makedirs(OUT_DIR, exist_ok=True)

def folder_bytes():
    return sum(os.path.getsize(os.path.join(OUT_DIR, f))
               for f in os.listdir(OUT_DIR))

def random_titles(n=50):
    r = requests.get(API, headers=HEADERS, timeout=30, params={
        'action': 'query', 'list': 'random',
        'rnnamespace': 0, 'rnlimit': n, 'format': 'json'})
    r.raise_for_status()
    return [p['title'] for p in r.json()['query']['random']]

target = TARGET_MB * 1_000_000
total  = folder_bytes()
new    = 0
print(f'starting at {total/1e6:.1f} MB, target {TARGET_MB} MB')

try:
    while total < target:
        for title in random_titles():
            if total >= target:
                break
            slug = quote(title.replace(' ', '_'), safe='')
            path = os.path.join(OUT_DIR, slug + '.html')
            if os.path.exists(path):
                continue
            try:
                r = requests.get(f'https://en.wikipedia.org/wiki/{slug}',
                                 headers=HEADERS, timeout=30)
                if r.status_code != 200:
                    continue
            except requests.RequestException:
                continue
            with open(path, 'w', encoding='utf-8') as f:
                f.write(r.text)
            total += os.path.getsize(path)
            new += 1
            if new % 25 == 0:
                print(f'{total/1e6:6.1f} MB   {new} new files', end='\r')
            time.sleep(DELAY)
except KeyboardInterrupt:
    print('\ninterrupted')

print(f'\ndone: {len(os.listdir(OUT_DIR))} files, {folder_bytes()/1e6:.1f} MB')

starting at 140.8 MB, target 200 MB
 196.2 MB   375 new files
done: 1359 files, 200.0 MB
